## Example 1: Basic WebSocket Server and Client

In [ ]:
import websockets
import asyncio
import json

# Example 1: Basic WebSocket server and client setup
print("Basic WebSocket server and client setup:")

# Simple echo server handler
async def echo_handler(websocket, path):
    """Simple echo server that returns received messages"""
    print(f"Client connected from {websocket.remote_address}")
    
    async for message in websocket:
        print(f"Received: {message}")
        # Echo the message back
        await websocket.send(f"Echo: {message}")

print("WebSocket echo server handler defined")

# Test client connection function
async def test_client(uri, message):
    """Test client that connects and sends a message"""
    try:
        async with websockets.connect(uri) as websocket:
            print(f"Connected to {uri}")
            
            # Send message
            await websocket.send(message)
            print(f"Sent: {message}")
            
            # Receive response
            response = await websocket.recv()
            print(f"Received: {response}")
            
            return response
            
    except Exception as e:
        print(f"Connection error: {e}")
        return None

print("WebSocket test client function defined")

# Show websockets package info
print(f"\nWebSockets package info:")
print(f"Version: {getattr(websockets, '__version__', 'Unknown')}")
print(f"Main functions: serve, connect")
print(f"Protocol support: WebSocket RFC 6455")
print("Async/await based WebSocket implementation")

## Example 2: Message Broadcasting Server

In [ ]:
import websockets
import asyncio
import json

# Example 2: WebSocket server with message broadcasting
print("WebSocket server with message broadcasting:")

class BroadcastServer:
    """WebSocket server that broadcasts messages to all connected clients"""
    
    def __init__(self):
        self.clients = set()
        self.message_history = []
    
    async def register_client(self, websocket):
        """Register a new client connection"""
        self.clients.add(websocket)
        print(f"Client registered. Total clients: {len(self.clients)}")
        
        # Send message history to new client
        for message in self.message_history[-5:]:  # Send last 5 messages
            await websocket.send(json.dumps({"type": "history", "data": message}))
    
    async def unregister_client(self, websocket):
        """Unregister a client connection"""
        self.clients.discard(websocket)
        print(f"Client unregistered. Total clients: {len(self.clients)}")
    
    async def broadcast_message(self, message, sender=None):
        """Broadcast message to all connected clients"""
        if self.clients:
            # Add to message history
            self.message_history.append(message)
            if len(self.message_history) > 100:  # Keep only last 100 messages
                self.message_history = self.message_history[-100:]
            
            # Broadcast to all clients
            message_data = json.dumps({"type": "broadcast", "data": message})
            disconnected = set()
            
            for client in self.clients:
                if client != sender:  # Don't echo back to sender
                    try:
                        await client.send(message_data)
                    except websockets.exceptions.ConnectionClosed:
                        disconnected.add(client)
            
            # Remove disconnected clients
            for client in disconnected:
                self.clients.discard(client)
            
            print(f"Broadcasted message to {len(self.clients)} clients")
    
    async def handle_client(self, websocket, path):
        """Handle individual client connections"""
        await self.register_client(websocket)
        
        try:
            async for message in websocket:
                try:
                    data = json.loads(message)
                    if data.get("type") == "message":
                        await self.broadcast_message(data.get("data"), sender=websocket)
                except json.JSONDecodeError:
                    # Handle plain text messages
                    await self.broadcast_message(message, sender=websocket)
                    
        except websockets.exceptions.ConnectionClosed:
            pass
        finally:
            await self.unregister_client(websocket)

# Create broadcast server instance
broadcast_server = BroadcastServer()
print("Broadcast server instance created")

# Test message simulation
async def simulate_messages():
    """Simulate some messages for testing"""
    test_messages = [
        "Hello from server!",
        "Broadcasting test message",
        "WebSocket server is working"
    ]
    
    for msg in test_messages:
        await broadcast_server.broadcast_message(msg)
        print(f"Simulated message: {msg}")

# Run simulation
await simulate_messages()

print(f"\nBroadcast server features:")
print("- Multi-client connection management")
print("- Message history and replay")
print("- JSON and plain text message support")
print("- Automatic client cleanup on disconnect")

## Example 3: Advanced Client with Reconnection

In [ ]:
import websockets
import asyncio
import json
import time

# Example 3: WebSocket client with connection management and reconnection
print("WebSocket client with connection management:")

class WebSocketClient:
    """Advanced WebSocket client with reconnection and message queuing"""
    
    def __init__(self, uri):
        self.uri = uri
        self.websocket = None
        self.connected = False
        self.message_queue = asyncio.Queue()
        self.reconnect_interval = 5
        self.max_reconnect_attempts = 3
    
    async def connect(self):
        """Establish WebSocket connection"""
        try:
            self.websocket = await websockets.connect(self.uri)
            self.connected = True
            print(f"Connected to {self.uri}")
            return True
        except Exception as e:
            print(f"Connection failed: {e}")
            return False
    
    async def disconnect(self):
        """Gracefully disconnect"""
        if self.websocket:
            await self.websocket.close()
            self.connected = False
            print("Disconnected from server")
    
    async def send_message(self, message):
        """Send message with connection check"""
        if not self.connected or not self.websocket:
            print("Not connected, queueing message")
            await self.message_queue.put(message)
            return False
        
        try:
            if isinstance(message, dict):
                message = json.dumps(message)
            
            await self.websocket.send(message)
            print(f"Sent: {message}")
            return True
            
        except websockets.exceptions.ConnectionClosed:
            print("Connection lost during send")
            self.connected = False
            await self.message_queue.put(message)
            return False
    
    async def receive_messages(self):
        """Listen for incoming messages"""
        try:
            async for message in self.websocket:
                try:
                    data = json.loads(message)
                    print(f"Received JSON: {data}")
                    yield data
                except json.JSONDecodeError:
                    print(f"Received text: {message}")
                    yield message
                    
        except websockets.exceptions.ConnectionClosed:
            print("Connection closed by server")
            self.connected = False
    
    async def process_message_queue(self):
        """Process queued messages when connected"""
        while not self.message_queue.empty():
            try:
                message = self.message_queue.get_nowait()
                success = await self.send_message(message)
                if success:
                    self.message_queue.task_done()
            except asyncio.QueueEmpty:
                break
    
    async def auto_reconnect(self):
        """Automatic reconnection with exponential backoff"""
        attempt = 0
        while attempt < self.max_reconnect_attempts:
            if await self.connect():
                await self.process_message_queue()
                return True
            
            attempt += 1
            wait_time = self.reconnect_interval * (2 ** attempt)
            print(f"Reconnect attempt {attempt} failed, waiting {wait_time}s")
            await asyncio.sleep(wait_time)
        
        print("Max reconnection attempts reached")
        return False

print("WebSocket client class defined")

# Test the client against a loopback echo server started right here (no public endpoint: the test needs no network)
async def echo(websocket):
    async for message in websocket:
        await websocket.send(message)

async def test_websocket_client():
    """Test WebSocket client functionality"""
    async with websockets.serve(echo, "127.0.0.1", 0) as server:
        port = server.sockets[0].getsockname()[1]
        client = WebSocketClient(f"ws://127.0.0.1:{port}")
        
        # Test connection: the server is local and ours, so failing to connect is a failure, not a message
        assert await client.connect(), f"could not connect to the local echo server on port {port}"
        
        # Send test messages; the echo server must return each one unchanged
        assert await client.send_message("Hello WebSocket!")
        reply = await client.websocket.recv()
        assert reply == "Hello WebSocket!", f"echo returned {reply!r}"
        payload = {"type": "test", "data": "JSON message", "timestamp": time.time()}
        assert await client.send_message(payload)
        reply = await client.websocket.recv()
        assert json.loads(reply) == payload, f"echo returned {reply!r}"
        print("Client test completed - both messages echoed back")
        
        await client.disconnect()
        
        # Message queuing + automatic reconnection: a message sent while disconnected is queued, and auto_reconnect()
        # must reconnect and flush it (a failed attempt prints, backs off, and ends in False)
        assert not await client.send_message("queued while offline")
        assert await client.auto_reconnect(), "auto_reconnect() gave up on the local echo server"
        reply = await client.websocket.recv()
        assert reply == "queued while offline", f"echo returned {reply!r}"
        print("Queued message delivered after reconnect")
        await client.disconnect()

# Run the test
try:
    await test_websocket_client()
except Exception as e:
    print(f"Client test error: {e}")
    raise AssertionError(
        f"websockets Example 3 failed: {type(e).__name__}: {e}"
    ) from e

print(f"\nClient features:")
print("- Automatic reconnection with backoff")
print("- Message queuing during disconnection")  
print("- JSON and text message support")
print("- Graceful connection management")

## Example 4: Authentication and Security Patterns

In [ ]:
import websockets
import asyncio
import json

# Example 4: WebSocket authentication and security patterns
print("WebSocket authentication and security patterns:")

class SecureWebSocketServer:
    """WebSocket server with authentication and security features"""
    
    def __init__(self):
        self.authenticated_clients = {}
        self.api_keys = {"test-key-123": {"user": "alice", "permissions": ["read", "write"]}}
        self.rate_limits = {}  # Simple rate limiting
    
    async def authenticate_client(self, websocket, auth_message):
        """Authenticate client using API key"""
        try:
            auth_data = json.loads(auth_message)
            api_key = auth_data.get("api_key")
            
            if api_key in self.api_keys:
                user_info = self.api_keys[api_key]
                self.authenticated_clients[websocket] = user_info
                
                await websocket.send(json.dumps({
                    "type": "auth_success",
                    "user": user_info["user"],
                    "permissions": user_info["permissions"]
                }))
                print(f"Client authenticated as {user_info['user']}")
                return True
            else:
                await websocket.send(json.dumps({
                    "type": "auth_failed",
                    "message": "Invalid API key"
                }))
                return False
                
        except (json.JSONDecodeError, KeyError) as e:
            await websocket.send(json.dumps({
                "type": "auth_error",
                "message": "Invalid authentication format"
            }))
            return False
    
    def check_rate_limit(self, websocket, max_messages_per_minute=60):
        """Simple rate limiting check"""
        import time
        current_time = time.time()
        
        if websocket not in self.rate_limits:
            self.rate_limits[websocket] = []
        
        # Remove old entries (older than 1 minute)
        self.rate_limits[websocket] = [
            timestamp for timestamp in self.rate_limits[websocket]
            if current_time - timestamp < 60
        ]
        
        if len(self.rate_limits[websocket]) >= max_messages_per_minute:
            return False
        
        self.rate_limits[websocket].append(current_time)
        return True
    
    def check_permission(self, websocket, required_permission):
        """Check if client has required permission"""
        if websocket not in self.authenticated_clients:
            return False
        
        user_permissions = self.authenticated_clients[websocket].get("permissions", [])
        return required_permission in user_permissions
    
    async def handle_secure_client(self, websocket, path):
        """Handle client with authentication and security checks"""
        authenticated = False
        
        try:
            # Wait for authentication message
            auth_message = await asyncio.wait_for(websocket.recv(), timeout=30)
            authenticated = await self.authenticate_client(websocket, auth_message)
            
            if not authenticated:
                await websocket.close(code=4001, reason="Authentication failed")
                return
            
            # Handle authenticated messages
            async for message in websocket:
                # Rate limiting check
                if not self.check_rate_limit(websocket):
                    await websocket.send(json.dumps({
                        "type": "error",
                        "message": "Rate limit exceeded"
                    }))
                    continue
                
                try:
                    data = json.loads(message)
                    message_type = data.get("type")
                    
                    if message_type == "write_data":
                        if self.check_permission(websocket, "write"):
                            await websocket.send(json.dumps({
                                "type": "write_success",
                                "data": data.get("data")
                            }))
                        else:
                            await websocket.send(json.dumps({
                                "type": "permission_denied",
                                "message": "Write permission required"
                            }))
                    
                    elif message_type == "read_data":
                        if self.check_permission(websocket, "read"):
                            await websocket.send(json.dumps({
                                "type": "read_success",
                                "data": "Sample secure data"
                            }))
                        else:
                            await websocket.send(json.dumps({
                                "type": "permission_denied", 
                                "message": "Read permission required"
                            }))
                    
                except json.JSONDecodeError:
                    await websocket.send(json.dumps({
                        "type": "error",
                        "message": "Invalid JSON format"
                    }))
                    
        except asyncio.TimeoutError:
            await websocket.close(code=4002, reason="Authentication timeout")
        except websockets.exceptions.ConnectionClosed:
            pass
        finally:
            # Cleanup
            if websocket in self.authenticated_clients:
                del self.authenticated_clients[websocket]
            if websocket in self.rate_limits:
                del self.rate_limits[websocket]

# Create secure server instance
secure_server = SecureWebSocketServer()
print("Secure WebSocket server created")

# Test authentication simulation
async def simulate_auth_test():
    """Simulate authentication testing"""
    
    # Valid authentication message
    valid_auth = json.dumps({"api_key": "test-key-123"})
    print(f"Valid auth message prepared: {valid_auth}")
    
    # Invalid authentication message
    invalid_auth = json.dumps({"api_key": "invalid-key"})
    print(f"Invalid auth message prepared: {invalid_auth}")
    
    print("Authentication simulation completed")

await simulate_auth_test()

print(f"\nSecurity features:")
print("- API key authentication")
print("- Permission-based access control") 
print("- Rate limiting protection")
print("- Input validation and sanitization")
print("- Graceful error handling and cleanup")

## Example 5: Production Application with Monitoring

In [ ]:
import websockets
import asyncio
import json
import time

# Example 5: Production WebSocket application with monitoring
print("Production WebSocket application with monitoring:")

class ProductionWebSocketApp:
    """Production-ready WebSocket application with comprehensive monitoring"""
    
    def __init__(self):
        self.connections = {}
        self.stats = {
            "total_connections": 0,
            "active_connections": 0,
            "messages_sent": 0,
            "messages_received": 0,
            "errors": 0,
            "start_time": time.time()
        }
        self.health_check_interval = 30
        self.monitoring_enabled = True
    
    async def register_connection(self, websocket, user_id=None):
        """Register a new WebSocket connection"""
        connection_id = id(websocket)
        self.connections[connection_id] = {
            "websocket": websocket,
            "user_id": user_id,
            "connected_at": time.time(),
            "last_ping": time.time(),
            "messages_sent": 0,
            "messages_received": 0
        }
        
        self.stats["total_connections"] += 1
        self.stats["active_connections"] += 1
        
        print(f"Connection registered: {connection_id} (user: {user_id})")
        return connection_id
    
    async def unregister_connection(self, connection_id):
        """Unregister a WebSocket connection"""
        if connection_id in self.connections:
            del self.connections[connection_id]
            self.stats["active_connections"] -= 1
            print(f"Connection unregistered: {connection_id}")
    
    async def send_message(self, connection_id, message):
        """Send message to specific connection"""
        if connection_id not in self.connections:
            return False
        
        connection = self.connections[connection_id]
        try:
            if isinstance(message, dict):
                message = json.dumps(message)
            
            await connection["websocket"].send(message)
            connection["messages_sent"] += 1
            self.stats["messages_sent"] += 1
            return True
            
        except Exception as e:
            print(f"Error sending message: {e}")
            self.stats["errors"] += 1
            return False
    
    async def broadcast_message(self, message, exclude_connection=None):
        """Broadcast message to all connections"""
        if isinstance(message, dict):
            message = json.dumps(message)
        
        successful_sends = 0
        failed_connections = []
        
        for conn_id, connection in self.connections.items():
            if conn_id != exclude_connection:
                try:
                    await connection["websocket"].send(message)
                    connection["messages_sent"] += 1
                    successful_sends += 1
                except Exception:
                    failed_connections.append(conn_id)
        
        # Clean up failed connections
        for conn_id in failed_connections:
            await self.unregister_connection(conn_id)
        
        self.stats["messages_sent"] += successful_sends
        return successful_sends
    
    def get_health_status(self):
        """Get application health status"""
        uptime = time.time() - self.stats["start_time"]
        
        return {
            "status": "healthy" if self.stats["active_connections"] >= 0 else "unhealthy",
            "uptime_seconds": round(uptime, 2),
            "active_connections": self.stats["active_connections"],
            "total_connections": self.stats["total_connections"],
            "messages_per_second": round(self.stats["messages_sent"] / uptime, 2) if uptime > 0 else 0,
            "error_rate": round(self.stats["errors"] / max(1, self.stats["messages_sent"]) * 100, 2),
            "memory_usage": f"{len(self.connections)} connections in memory"
        }
    
    def get_connection_stats(self):
        """Get detailed connection statistics"""
        if not self.connections:
            return {"message": "No active connections"}
        
        stats = []
        current_time = time.time()
        
        for conn_id, connection in self.connections.items():
            duration = current_time - connection["connected_at"]
            stats.append({
                "connection_id": conn_id,
                "user_id": connection.get("user_id", "anonymous"),
                "duration_seconds": round(duration, 2),
                "messages_sent": connection["messages_sent"],
                "messages_received": connection["messages_received"]
            })
        
        return stats
    
    async def handle_production_client(self, websocket, path):
        """Handle client connection with full production features"""
        connection_id = await self.register_connection(websocket)
        
        try:
            # Send welcome message
            await self.send_message(connection_id, {
                "type": "welcome",
                "connection_id": connection_id,
                "server_info": "Production WebSocket Server v1.0"
            })
            
            async for message in websocket:
                self.connections[connection_id]["messages_received"] += 1
                self.stats["messages_received"] += 1
                
                try:
                    data = json.loads(message)
                    message_type = data.get("type")
                    
                    if message_type == "ping":
                        await self.send_message(connection_id, {"type": "pong", "timestamp": time.time()})
                        self.connections[connection_id]["last_ping"] = time.time()
                    
                    elif message_type == "broadcast":
                        sent_count = await self.broadcast_message(data.get("message"), exclude_connection=connection_id)
                        await self.send_message(connection_id, {
                            "type": "broadcast_result",
                            "sent_to": sent_count
                        })
                    
                    elif message_type == "health":
                        health = self.get_health_status()
                        await self.send_message(connection_id, {
                            "type": "health_status",
                            "data": health
                        })
                
                except json.JSONDecodeError:
                    await self.send_message(connection_id, {
                        "type": "error",
                        "message": "Invalid JSON format"
                    })
                    
        except websockets.exceptions.ConnectionClosed:
            print(f"Connection {connection_id} closed")
        except Exception as e:
            print(f"Error handling client {connection_id}: {e}")
            self.stats["errors"] += 1
        finally:
            await self.unregister_connection(connection_id)

# Create production application
app = ProductionWebSocketApp()
print("Production WebSocket application created")

# Simulate some activity for demonstration
async def simulate_production_activity():
    """Simulate production application activity"""
    
    # Simulate connection registration
    class MockWebSocket:
        def __init__(self, name):
            self.name = name
        async def send(self, message):
            print(f"Mock send to {self.name}: {message[:50]}...")
    
    # Register mock connections
    ws1 = MockWebSocket("client1")
    ws2 = MockWebSocket("client2")
    
    conn1 = await app.register_connection(ws1, "user123")
    conn2 = await app.register_connection(ws2, "user456")
    
    # Simulate message sending
    await app.send_message(conn1, {"type": "test", "data": "Hello client 1"})
    await app.send_message(conn2, {"type": "test", "data": "Hello client 2"})
    
    # Test broadcasting
    broadcast_count = await app.broadcast_message({"type": "announcement", "data": "Server announcement"})
    print(f"Broadcast sent to {broadcast_count} clients")
    
    # Show health status
    health = app.get_health_status()
    print(f"Health status: {health}")
    
    # Show connection stats
    conn_stats = app.get_connection_stats()
    print(f"Connection stats: {len(conn_stats)} active connections")
    
    # Cleanup
    await app.unregister_connection(conn1)
    await app.unregister_connection(conn2)

await simulate_production_activity()

print(f"\nProduction features:")
print("- Comprehensive connection management")
print("- Real-time statistics and monitoring")
print("- Health check endpoints")
print("- Broadcasting and unicast messaging")
print("- Error tracking and resilience")
print("- Performance metrics and analytics")